# Improved Binary Ensemble Training

This notebook trains a **binary ONNX ensemble** (Cancer vs Non-cancer) using:

## Data Sources (~4,800 images)
1. **Oral-Image-small-binary-dataset** (~326 images, original only) - NEW
2. **oral_cavity_binary_dataset** (~1,634 images)
3. **oral_cavity_5class_dataset** (~2,838 images, mapped to binary)

**Note:** We only use original images from small-binary-dataset (not pre-augmented) to avoid double-augmentation issues when Albumentations is applied during training.

## Improvements over previous binary_ensemble_training.ipynb
- Additional data source (small binary dataset)
- Advanced Albumentations augmentation (medical-grade)
- Cosine annealing LR scheduler
- Tunable class weights for cancer sensitivity
- New frozen test set TC from small binary dataset
- Detailed agreement analysis vs 5-class model

## Architecture
- **Backbone:** MobileNetV3Large (same as 5-class model for consistency)
- **Output:** 3-model ensemble, averaged at inference
- **Export:** ONNX (opset 18, single-file)

## 1. Configuration

In [1]:
# Configuration
import os
from pathlib import Path
from datetime import datetime
import json

# Dataset paths
DATA_DIR_5CLASS = os.path.expanduser("~/dev/oral_cavity_5class_dataset")
DATA_DIR_BINARY = os.path.expanduser("~/dev/oral_cavity_binary_dataset")
DATA_DIR_SMALL_BINARY = os.path.expanduser("~/dev/oral-images-dataset/Oral-Image-small-binary-dataset")

NOTEBOOK_DIR = Path(".").resolve()
FROZEN_DIR = NOTEBOOK_DIR / "frozen_test_sets"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = os.environ.get("EXPERIMENT_NAME") or f"improved_run_{timestamp}"

# Training hyperparameters
DEFAULT_CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "learning_rate": 1e-4,
    "num_epochs": 50,
    "patience": 10,
    "val_split": 0.15,
    "num_workers": 0,
    "seed": 42,
    
    # Improved settings
    "augmentation": "medical",  # basic, strong, or medical
    "scheduler": "cosine",  # step, cosine, or none
    "cancer_weight_multiplier": 1.5,  # Increase sensitivity for cancer class
    "label_smoothing": 0.05,
    "dropout_rates": [0.3, 0.1, 0.2],  # For ensemble models 1, 2, 3
    
    # Threshold for binary classification
    "binary_threshold": 0.5,
}

# Allow config override from environment
config_path = os.environ.get("EXPERIMENT_CONFIG")
if config_path and Path(config_path).exists():
    with open(config_path, "r") as f:
        overrides = json.load(f)
    DEFAULT_CONFIG.update(overrides)

CONFIG = DEFAULT_CONFIG
run_name = CONFIG.get("run_name", run_name)

# Output directory
OUTPUT_DIR = NOTEBOOK_DIR / f"results/improved_binary_ensemble/{run_name}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)
(OUTPUT_DIR / "onnx").mkdir(exist_ok=True)

# Save config
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({
        "data_dir_5class": DATA_DIR_5CLASS,
        "data_dir_binary": DATA_DIR_BINARY,
        "data_dir_small_binary": DATA_DIR_SMALL_BINARY,
        **CONFIG
    }, f, indent=2)

print("Configuration")
print("=" * 60)
print(f"5-class data: {DATA_DIR_5CLASS}")
print(f"Binary data: {DATA_DIR_BINARY}")
print(f"Small binary data: {DATA_DIR_SMALL_BINARY}")
print(f"Frozen sets: {FROZEN_DIR}")
print(f"Output: {OUTPUT_DIR}")
print()
print(f"Epochs: {CONFIG['num_epochs']}, Patience: {CONFIG['patience']}")
print(f"LR: {CONFIG['learning_rate']}, Scheduler: {CONFIG['scheduler']}")
print(f"Augmentation: {CONFIG['augmentation']}")
print(f"Cancer weight multiplier: {CONFIG['cancer_weight_multiplier']}")
print(f"Label smoothing: {CONFIG['label_smoothing']}")

Configuration
5-class data: /Users/alokdwivedi/dev/oral_cavity_5class_dataset
Binary data: /Users/alokdwivedi/dev/oral_cavity_binary_dataset
Small binary data: /Users/alokdwivedi/dev/oral-images-dataset/Oral-Image-small-binary-dataset
Frozen sets: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642

Epochs: 50, Patience: 10
LR: 0.0001, Scheduler: cosine
Augmentation: medical
Cancer weight multiplier: 1.5
Label smoothing: 0.05


## 2. Setup and Device

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix, 
    brier_score_loss, classification_report
)
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("Warning: albumentations not installed. Using basic augmentation.")
    print("Install with: pip install albumentations")

try:
    import onnx
    import onnxruntime as ort
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False

print(f"PyTorch: {torch.__version__}")
if HAS_ALBUMENTATIONS:
    print(f"Albumentations: {A.__version__}")
if HAS_ONNX:
    print(f"ONNX: {onnx.__version__}, Runtime: {ort.__version__}")

Install with: pip install albumentations
PyTorch: 2.10.0
ONNX: 1.20.1, Runtime: 1.23.2


In [3]:
# Device setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using Apple Silicon GPU (MPS): {device}")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {device}")
else:
    device = torch.device("cpu")
    print(f"Using CPU: {device}")

# Set seed for reproducibility
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

Using Apple Silicon GPU (MPS): mps


## 3. Build Frozen Test Sets (v2)

In [4]:
import sys
sys.path.insert(0, str(NOTEBOOK_DIR))

from frozen_test_sets import (
    build_and_save_frozen_sets_v2,
    load_frozen_sets_v2,
    get_all_train_val_for_training,
    get_t5_binary_paths_labels,
    get_tb_paths_labels,
    get_tc_paths_labels,
    get_tm_paths_labels_and_sources_v2,
)

# Build frozen test sets v2 (includes TC)
T5, TB, TC, TM, meta = build_and_save_frozen_sets_v2(
    data_dir_5class=DATA_DIR_5CLASS,
    data_dir_binary=DATA_DIR_BINARY,
    data_dir_small_binary=DATA_DIR_SMALL_BINARY,
    out_dir=FROZEN_DIR,
)

print("\nFrozen test set summary:")
print(f"  T5: {len(T5)} (5-class test, binary-eligible: {meta['T5_binary_eligible']})")
print(f"  TB: {len(TB)} (binary dataset)")
print(f"  TC: {len(TC)} (small binary dataset)")
print(f"  TM: {len(TM)} (merged)")

Building frozen test sets v2...
  5-class dataset: /Users/alokdwivedi/dev/oral_cavity_5class_dataset
  Binary dataset: /Users/alokdwivedi/dev/oral_cavity_binary_dataset
  Small binary dataset: /Users/alokdwivedi/dev/oral-images-dataset/Oral-Image-small-binary-dataset

Frozen test sets v2 saved to /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
  T5: 534 samples (5-class test)
  TB: 246 samples (binary dataset test)
  TC: 49 samples (small binary dataset test)
  TM: 779 samples (mixed, all sources)
  binary_train_val_for_training: 1388 paths
  small_binary_train_val_for_training: 274 paths

Frozen test set summary:
  T5: 534 (5-class test, binary-eligible: 484)
  TB: 246 (binary dataset)
  TC: 49 (small binary dataset)
  TM: 779 (merged)


## 4. Load and Merge Training Data

In [5]:
def collect_images(folder, exts=("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")):
    """Collect all images from a folder."""
    folder = Path(folder)
    out = []
    if folder.exists():
        for ext in exts:
            out.extend([str(p) for p in folder.glob(ext)])
    return out

# Get binary train/val pools from frozen_test_sets (excludes TB and TC)
binary_train_paths, binary_train_labels, binary_train_sources = get_all_train_val_for_training(FROZEN_DIR)

# Get 5-class train+val mapped to binary (exclude Unknown)
five_dir = Path(DATA_DIR_5CLASS)
five_cancer, five_non = [], []
for split in ["train", "val"]:
    for c in ["OCA", "OPMD"]:
        five_cancer.extend(collect_images(five_dir / split / c))
    for c in ["Benign", "Healthy"]:
        five_non.extend(collect_images(five_dir / split / c))

# Merge all sources (dedup by path)
all_paths_set = set()
all_cancer, all_non = [], []

# Add binary sources (B and C)
for path, label in zip(binary_train_paths, binary_train_labels):
    if path not in all_paths_set:
        all_paths_set.add(path)
        if label == 1:
            all_cancer.append(path)
        else:
            all_non.append(path)

# Add 5-class (source A)
for p in five_cancer:
    if p not in all_paths_set:
        all_paths_set.add(p)
        all_cancer.append(p)
for p in five_non:
    if p not in all_paths_set:
        all_paths_set.add(p)
        all_non.append(p)

all_paths = all_cancer + all_non
all_labels = [1] * len(all_cancer) + [0] * len(all_non)

print("Merged training pool:")
print(f"  Cancer (1): {len(all_cancer)}")
print(f"  Non-cancer (0): {len(all_non)}")
print(f"  Total: {len(all_paths)}")
print(f"\nClass balance: {len(all_cancer)/len(all_paths)*100:.1f}% cancer, {len(all_non)/len(all_paths)*100:.1f}% non-cancer")

Merged training pool:
  Cancer (1): 2320
  Non-cancer (0): 2180
  Total: 4500

Class balance: 51.6% cancer, 48.4% non-cancer


## 5. Train/Val Split

In [6]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=CONFIG["val_split"],
    random_state=CONFIG["seed"],
    stratify=all_labels,
)

print(f"Train: {len(train_paths)} ({sum(train_labels)} cancer, {len(train_labels)-sum(train_labels)} non-cancer)")
print(f"Val: {len(val_paths)} ({sum(val_labels)} cancer, {len(val_labels)-sum(val_labels)} non-cancer)")

Train: 3825 (1972 cancer, 1853 non-cancer)
Val: 675 (348 cancer, 327 non-cancer)


## 6. Augmentation Pipeline (Albumentations)

In [7]:
# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_augmentation_pipeline(mode="medical", image_size=(224, 224)):
    """Get augmentation pipeline based on mode."""
    
    if HAS_ALBUMENTATIONS and mode in ["medical", "strong"]:
        if mode == "medical":
            # Medical-grade augmentation for oral cavity images
            return A.Compose([
                A.Resize(image_size[0], image_size[1]),
                
                # Geometric transformations
                A.RandomRotate90(p=0.4),
                A.Flip(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
                
                # Color/contrast (oral cavity specific)
                A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
                A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=15, val_shift_limit=12, p=0.4),
                A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.4),  # Enhance lesion visibility
                
                # Consumer camera simulation
                A.OneOf([
                    A.GaussNoise(var_limit=(10.0, 40.0), p=0.3),
                    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.3), p=0.2),
                ], p=0.25),
                
                # Quality degradation (real-world conditions)
                A.OneOf([
                    A.MotionBlur(blur_limit=5, p=0.3),
                    A.GaussianBlur(blur_limit=3, p=0.3),
                ], p=0.2),
                A.ImageCompression(quality_lower=70, quality_upper=95, p=0.15),
                
                # Normalize and convert to tensor
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ])
        else:  # strong
            return A.Compose([
                A.Resize(image_size[0], image_size[1]),
                A.RandomResizedCrop(image_size[0], image_size[1], scale=(0.7, 1.0)),
                A.HorizontalFlip(p=0.5),
                A.RandomRotate90(p=0.4),
                A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=20, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
                A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=0.5),
                A.GaussianBlur(blur_limit=5, p=0.3),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ])
    else:
        # Basic torchvision transforms
        return None

def get_val_transform(image_size=(224, 224)):
    """Validation transform (no augmentation)."""
    if HAS_ALBUMENTATIONS:
        return A.Compose([
            A.Resize(image_size[0], image_size[1]),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return None

# Get augmentation pipelines
train_aug = get_augmentation_pipeline(CONFIG["augmentation"], CONFIG["image_size"])
val_aug = get_val_transform(CONFIG["image_size"])

print(f"Using augmentation: {CONFIG['augmentation']}")
print(f"Albumentations available: {HAS_ALBUMENTATIONS}")

Using augmentation: medical
Albumentations available: False


In [8]:
# Fallback torchvision transforms (if albumentations not available)
basic_train_transform = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

basic_val_transform = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class BinaryDataset(Dataset):
    """Binary classification dataset with optional Albumentations support."""
    
    def __init__(self, paths, labels, albu_transform=None, torch_transform=None):
        self.paths = paths
        self.labels = labels
        self.albu_transform = albu_transform
        self.torch_transform = torch_transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        
        img = Image.open(path).convert("RGB")
        
        if self.albu_transform is not None:
            img_np = np.array(img)
            augmented = self.albu_transform(image=img_np)
            img = augmented["image"]
        elif self.torch_transform is not None:
            img = self.torch_transform(img)
        
        return img, label


# Create datasets
if train_aug is not None:
    train_ds = BinaryDataset(train_paths, train_labels, albu_transform=train_aug)
    val_ds = BinaryDataset(val_paths, val_labels, albu_transform=val_aug)
else:
    train_ds = BinaryDataset(train_paths, train_labels, torch_transform=basic_train_transform)
    val_ds = BinaryDataset(val_paths, val_labels, torch_transform=basic_val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=False,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 120, Val batches: 22


## 7. Model Architecture (MobileNetV3Large)

In [9]:
NUM_CLASSES = 2  # 0 = non-cancer, 1 = cancer

class ConservativeMobileNetV3(nn.Module):
    """Conservative MobileNetV3Large - same architecture as 5-class model."""
    
    def __init__(self, num_classes=2, dropout_rate=0.2):
        super().__init__()
        try:
            self.backbone = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        except AttributeError:
            self.backbone = models.mobilenet_v3_large(pretrained=True)
        
        self.backbone.classifier = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(960, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.backbone.features(x)
        x = self.classifier(x)
        return x


# Ensemble variants with different dropout rates
class EnsembleModel1(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][0])


class EnsembleModel2(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][1])


class EnsembleModel3(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][2])
        # Slightly different architecture for diversity
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(960, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes),
        )


ENSEMBLE_CLASSES = [EnsembleModel1, EnsembleModel2, EnsembleModel3]
print(f"Model classes defined (num_classes={NUM_CLASSES})")
print(f"Dropout rates: {CONFIG['dropout_rates']}")

Model classes defined (num_classes=2)
Dropout rates: [0.3, 0.1, 0.2]


## 8. Class Weights

In [10]:
# Compute class weights for imbalanced data
classes = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=classes, y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32)

# Apply cancer weight multiplier for increased sensitivity
cancer_multiplier = float(CONFIG.get("cancer_weight_multiplier", 1.0))
if cancer_multiplier != 1.0:
    class_weights = class_weights.clone()
    class_weights[1] = class_weights[1] * cancer_multiplier
    print(f"Applied cancer weight multiplier: {cancer_multiplier}")

print(f"Class weights: non-cancer={class_weights[0]:.4f}, cancer={class_weights[1]:.4f}")

Applied cancer weight multiplier: 1.5
Class weights: non-cancer=1.0321, cancer=1.4547


## 9. Training Loop

In [11]:
def train_ensemble_model(
    model_class,
    model_name,
    train_loader,
    val_loader,
    device,
    config,
    class_weights=None,
):
    """Train one binary ensemble model with improved training loop."""
    
    model = model_class(num_classes=NUM_CLASSES).to(device)
    
    # Loss with label smoothing
    label_smoothing = config.get("label_smoothing", 0.0)
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(device),
            label_smoothing=label_smoothing,
        )
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    
    optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])
    
    # Scheduler
    scheduler_type = config.get("scheduler", "step")
    if scheduler_type == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=config["num_epochs"], eta_min=1e-6
        )
    elif scheduler_type == "step":
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    else:
        scheduler = None
    
    best_val_acc = 0.0
    best_val_auc = 0.0
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_auc": [], "lr": []}
    
    for epoch in range(config["num_epochs"]):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for data, target in train_loader:
            data, target = data.to(device), target.to(device).long()
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, pred = out.max(1)
            train_total += target.size(0)
            train_correct += pred.eq(target).sum().item()
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_probs = []
        val_targets = []
        
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device).long()
                out = model(data)
                probs = torch.softmax(out, dim=1)
                _, pred = out.max(1)
                
                val_total += target.size(0)
                val_correct += pred.eq(target).sum().item()
                val_probs.extend(probs[:, 1].cpu().numpy())
                val_targets.extend(target.cpu().numpy())
        
        val_acc = 100.0 * val_correct / val_total
        val_auc = roc_auc_score(val_targets, val_probs)
        current_lr = optimizer.param_groups[0]["lr"]
        
        history["train_loss"].append(train_loss / len(train_loader))
        history["train_acc"].append(100.0 * train_correct / train_total)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        history["lr"].append(current_lr)
        
        # Best model selection based on AUC (more robust than accuracy)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{config['num_epochs']}  "
                  f"loss={train_loss/len(train_loader):.4f}  "
                  f"train_acc={100.*train_correct/train_total:.2f}%  "
                  f"val_acc={val_acc:.2f}%  "
                  f"val_auc={val_auc:.4f}  "
                  f"lr={current_lr:.2e}")
        
        if patience_counter >= config["patience"]:
            print(f"  Early stop at epoch {epoch+1}")
            break
        
        if scheduler is not None:
            scheduler.step()
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return model, best_val_acc, best_val_auc, history

In [12]:
print("Training binary ensemble (3 models)")
print("=" * 60)

trained_models = []
all_histories = []

for i, model_class in enumerate(ENSEMBLE_CLASSES):
    name = f"ensemble_model_{i+1}"
    print(f"\n[{i+1}/3] {name}")
    
    model, val_acc, val_auc, history = train_ensemble_model(
        model_class, name, train_loader, val_loader, device, CONFIG, class_weights=class_weights
    )
    
    path = OUTPUT_DIR / "models" / f"{name}.pth"
    torch.save(model.state_dict(), path)
    print(f"  Saved: {path}")
    print(f"  Best val_acc={val_acc:.2f}%, val_auc={val_auc:.4f}")
    
    trained_models.append(model)
    all_histories.append(history)

print("\nAll 3 models trained and saved")

Training binary ensemble (3 models)

[1/3] ensemble_model_1
  Epoch 1/50  loss=0.5798  train_acc=66.27%  val_acc=71.85%  val_auc=0.8322  lr=1.00e-04
  Epoch 5/50  loss=0.2921  train_acc=89.93%  val_acc=78.67%  val_auc=0.8852  lr=9.84e-05
  Epoch 10/50  loss=0.1899  train_acc=95.92%  val_acc=80.30%  val_auc=0.8791  lr=9.23e-05
  Epoch 15/50  loss=0.1669  train_acc=97.44%  val_acc=79.85%  val_auc=0.8831  lr=8.21e-05
  Epoch 20/50  loss=0.1527  train_acc=97.91%  val_acc=80.44%  val_auc=0.8895  lr=6.87e-05
  Early stop at epoch 21
  Saved: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/models/ensemble_model_1.pth
  Best val_acc=81.04%, val_auc=0.8896

[2/3] ensemble_model_2
  Epoch 1/50  loss=0.5683  train_acc=66.54%  val_acc=74.22%  val_auc=0.8394  lr=1.00e-04
  Epoch 5/50  loss=0.2763  train_acc=91.03%  val_acc=80.89%  val_auc=0.8886  lr=9.84e-05
  Epoch 10/50  loss=0.1809  train_acc=96.84% 

## 10. Convert to ONNX

In [13]:
def convert_to_onnx(model, onnx_path, device, input_shape=(1, 3, 224, 224)):
    """Convert PyTorch model to single-file ONNX."""
    model.eval()
    model.to(device)
    
    dummy = torch.randn(input_shape).to(device)
    
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        export_params=True,
        opset_version=18,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    )
    
    # Ensure single-file ONNX (no external data)
    if HAS_ONNX:
        onnx_model = onnx.load(str(onnx_path))
        onnx.save_model(onnx_model, str(onnx_path), save_as_external_data=False)
        # Remove any .onnx.data file if created
        data_file = Path(str(onnx_path) + ".data")
        if data_file.exists():
            data_file.unlink()
    
    return True


onnx_dir = OUTPUT_DIR / "onnx"
for i, model in enumerate(trained_models):
    name = f"ensemble_model_{i+1}"
    onnx_path = onnx_dir / f"{name}.onnx"
    convert_to_onnx(model, onnx_path, device)
    size_mb = onnx_path.stat().st_size / 1024 / 1024
    print(f"{name}.onnx ({size_mb:.2f} MB)")

print(f"\nONNX models saved to: {onnx_dir}")

W0205 20:57:33.927000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 20:57:33.927000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 20:57:33.928000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0205 20:57:33.928000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EnsembleModel1([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel1([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.


W0205 20:57:37.300000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 20:57:37.301000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 20:57:37.301000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0205 20:57:37.302000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


ensemble_model_1.onnx (12.61 MB)
[torch.onnx] Obtain model graph for `EnsembleModel2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.


W0205 20:57:40.430000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 20:57:40.430000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0205 20:57:40.431000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0205 20:57:40.431000 27781 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


ensemble_model_2.onnx (12.61 MB)
[torch.onnx] Obtain model graph for `EnsembleModel3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
ensemble_model_3.onnx (13.55 MB)

ONNX models saved to: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/onnx


## 11. Evaluation on Frozen Test Sets

In [14]:
def preprocess_for_onnx(path, image_size=(224, 224)):
    """Preprocess image for ONNX inference."""
    img = Image.open(path).convert("RGB").resize(image_size)
    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    return np.transpose(arr, (2, 0, 1))[np.newaxis, ...].astype(np.float32)


def evaluate_on_frozen_set(onnx_dir, paths, labels, set_name, ensemble_weights=[0.30, 0.40, 0.30]):
    """Evaluate binary ensemble on a frozen test set."""
    
    sessions = [
        ort.InferenceSession(str(onnx_dir / f"ensemble_model_{i}.onnx"))
        for i in [1, 2, 3]
    ]
    input_name = sessions[0].get_inputs()[0].name
    
    y_true = []
    y_prob = []
    y_pred = []
    
    for path, label in zip(paths, labels):
        x = preprocess_for_onnx(path)
        
        # Ensemble averaging
        logits = sum(
            sess.run(None, {input_name: x})[0] * w
            for sess, w in zip(sessions, ensemble_weights)
        )
        exp_logits = np.exp(logits - np.max(logits))
        prob_cancer = float(exp_logits[0, 1] / exp_logits.sum())
        
        y_true.append(label)
        y_prob.append(prob_cancer)
        y_pred.append(1 if prob_cancer >= 0.5 else 0)
    
    # Metrics
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = np.array(y_pred)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    accuracy = (tp + tn) / len(y_true)
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    
    return {
        "set_name": set_name,
        "n": len(y_true),
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "auroc": auroc,
        "auprc": auprc,
        "confusion": {"tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)},
        "y_true": y_true.tolist(),
        "y_prob": y_prob.tolist(),
        "y_pred": y_pred.tolist(),
    }

In [15]:
print("Evaluating on frozen test sets...")
print("=" * 60)

results = {}

# T5 (5-class test, binary-eligible)
t5_paths, t5_labels = get_t5_binary_paths_labels(FROZEN_DIR)
results["T5"] = evaluate_on_frozen_set(onnx_dir, t5_paths, t5_labels, "T5")

# TB (binary dataset)
tb_paths, tb_labels = get_tb_paths_labels(FROZEN_DIR)
results["TB"] = evaluate_on_frozen_set(onnx_dir, tb_paths, tb_labels, "TB")

# TC (small binary dataset)
tc_paths, tc_labels = get_tc_paths_labels(FROZEN_DIR)
if len(tc_paths) > 0:
    results["TC"] = evaluate_on_frozen_set(onnx_dir, tc_paths, tc_labels, "TC")

# TM (merged)
tm_paths, tm_labels, tm_sources = get_tm_paths_labels_and_sources_v2(FROZEN_DIR)
results["TM"] = evaluate_on_frozen_set(onnx_dir, tm_paths, tm_labels, "TM")

# Print summary
print(f"\n{'Set':<6} {'N':<8} {'Acc':<10} {'Sens':<10} {'Spec':<10} {'AUROC':<10} {'AUPRC':<10}")
print("-" * 70)
for name in ["T5", "TB", "TC", "TM"]:
    if name in results:
        r = results[name]
        print(f"{name:<6} {r['n']:<8} {r['accuracy']*100:.2f}%    "
              f"{r['sensitivity']*100:.2f}%    {r['specificity']*100:.2f}%    "
              f"{r['auroc']:.4f}    {r['auprc']:.4f}")

# Save results
with open(OUTPUT_DIR / "evaluation_results.json", "w") as f:
    # Remove numpy arrays for JSON serialization
    results_json = {k: {kk: vv for kk, vv in v.items() if kk not in ["y_true", "y_prob", "y_pred"]} 
                    for k, v in results.items()}
    json.dump(results_json, f, indent=2)
print(f"\nResults saved to: {OUTPUT_DIR / 'evaluation_results.json'}")

Evaluating on frozen test sets...

Set    N        Acc        Sens       Spec       AUROC      AUPRC     
----------------------------------------------------------------------
T5     484      79.55%    80.82%    78.24%    0.8863    0.8998
TB     246      93.09%    90.23%    96.46%    0.9800    0.9863
TC     49       89.80%    100.00%    80.00%    1.0000    1.0000
TM     779      84.47%    85.07%    83.82%    0.9277    0.9403

Results saved to: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/evaluation_results.json


## 12. Agreement Analysis (Binary vs 5-class)

In [16]:
# Load 5-class ensemble for comparison
REPO_ROOT = NOTEBOOK_DIR.parent.parent.parent
FIVECLASS_ONNX_DIR = REPO_ROOT / "assets"

FIVECLASS_NAMES = ["Benign", "Healthy", "OCA", "OPMD", "Unknown"]
CANCER_CLASS_INDICES = {2, 3}  # OCA, OPMD -> binary 1


def run_agreement_analysis(binary_onnx_dir, fiveclass_onnx_dir, paths, labels, set_name):
    """Compare binary ensemble predictions with 5-class ensemble."""
    
    # Check if 5-class models exist
    if not (fiveclass_onnx_dir / "ensemble_model_1.onnx").exists():
        print(f"Warning: 5-class ONNX models not found at {fiveclass_onnx_dir}")
        return None
    
    # Load models
    binary_sessions = [
        ort.InferenceSession(str(binary_onnx_dir / f"ensemble_model_{i}.onnx"))
        for i in [1, 2, 3]
    ]
    five_sessions = [
        ort.InferenceSession(str(fiveclass_onnx_dir / f"ensemble_model_{i}.onnx"))
        for i in [1, 2, 3]
    ]
    
    input_name_bin = binary_sessions[0].get_inputs()[0].name
    input_name_5 = five_sessions[0].get_inputs()[0].name
    weights = [0.30, 0.40, 0.30]
    
    agreements = 0
    disagreements = []
    
    for path, label in zip(paths, labels):
        x = preprocess_for_onnx(path)
        
        # Binary prediction
        logits_bin = sum(sess.run(None, {input_name_bin: x})[0] * w for sess, w in zip(binary_sessions, weights))
        exp_bin = np.exp(logits_bin - np.max(logits_bin))
        prob_cancer = float(exp_bin[0, 1] / exp_bin.sum())
        pred_bin = 1 if prob_cancer >= 0.5 else 0
        
        # 5-class prediction
        logits_5 = sum(sess.run(None, {input_name_5: x})[0] * w for sess, w in zip(five_sessions, weights))
        idx_5 = int(np.argmax(logits_5[0]))
        label_5 = FIVECLASS_NAMES[idx_5]
        pred_5_bin = 1 if idx_5 in CANCER_CLASS_INDICES else 0
        
        if pred_bin == pred_5_bin:
            agreements += 1
        else:
            disagreements.append({
                "path": path,
                "ground_truth": label,
                "binary_pred": pred_bin,
                "binary_prob": prob_cancer,
                "fiveclass_label": label_5,
                "fiveclass_binary": pred_5_bin,
            })
    
    total = len(paths)
    agreement_rate = agreements / total * 100
    
    return {
        "set_name": set_name,
        "total": total,
        "agreements": agreements,
        "disagreements_count": len(disagreements),
        "agreement_rate": agreement_rate,
        "disagreements": disagreements[:20],  # Limit to first 20 for display
    }

In [17]:
print("Agreement Analysis: Binary vs 5-class Ensemble")
print("=" * 60)

agreement_results = {}

for set_name, (paths, labels) in [
    ("T5", (t5_paths, t5_labels)),
    ("TB", (tb_paths, tb_labels)),
    ("TC", (tc_paths, tc_labels) if len(tc_paths) > 0 else ([], [])),
    ("TM", (tm_paths, tm_labels)),
]:
    if len(paths) == 0:
        continue
    
    result = run_agreement_analysis(onnx_dir, FIVECLASS_ONNX_DIR, paths, labels, set_name)
    if result:
        agreement_results[set_name] = result
        print(f"{set_name}: {result['agreements']}/{result['total']} agreed ({result['agreement_rate']:.1f}%)")

# Save agreement results
with open(OUTPUT_DIR / "agreement_analysis.json", "w") as f:
    json.dump(agreement_results, f, indent=2)

print(f"\nAgreement analysis saved to: {OUTPUT_DIR / 'agreement_analysis.json'}")

Agreement Analysis: Binary vs 5-class Ensemble
T5: 397/484 agreed (82.0%)
TB: 188/246 agreed (76.4%)
TC: 44/49 agreed (89.8%)
TM: 629/779 agreed (80.7%)

Agreement analysis saved to: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/agreement_analysis.json


In [18]:
# Show sample disagreements
if "TM" in agreement_results and agreement_results["TM"]["disagreements"]:
    print("\nSample disagreements (TM):")
    print("-" * 80)
    for d in agreement_results["TM"]["disagreements"][:10]:
        filename = Path(d["path"]).name
        print(f"  {filename[:35]:<35} GT={d['ground_truth']} Binary={d['binary_pred']} (P={d['binary_prob']:.3f}) "
              f"5-class={d['fiveclass_label']} (->bin={d['fiveclass_binary']})")


Sample disagreements (TM):
--------------------------------------------------------------------------------
  S-124-01.jpg                        GT=0 Binary=1 (P=0.849) 5-class=Benign (->bin=0)
  N-277-01.jpg                        GT=0 Binary=1 (P=0.825) 5-class=Benign (->bin=0)
  R-271-03.jpg                        GT=0 Binary=1 (P=0.793) 5-class=Healthy (->bin=0)
  R-139-01.jpg                        GT=0 Binary=1 (P=0.818) 5-class=Benign (->bin=0)
  R-132-08.jpg                        GT=0 Binary=1 (P=0.892) 5-class=Healthy (->bin=0)
  S-05-01.jpg                         GT=0 Binary=1 (P=0.841) 5-class=Benign (->bin=0)
  N-142-04.jpg                        GT=0 Binary=1 (P=0.692) 5-class=Benign (->bin=0)
  R-11-02.jpg                         GT=0 Binary=0 (P=0.211) 5-class=OPMD (->bin=1)
  S-50-01.jpg                         GT=0 Binary=1 (P=0.977) 5-class=Benign (->bin=0)
  N-001-04.jpg                        GT=0 Binary=0 (P=0.053) 5-class=OPMD (->bin=1)


### Copy agreed vs disagreed samples to folders

Copy up to 10 agreed and 10 disagreed samples per test set into `agreed-samples/` and `disagreed-samples/` for inspection. Run after the agreement analysis cell; requires `agreement_results` and path variables (e.g. `tm_paths`, `t5_paths`) to be in scope.

In [32]:
import shutil

agreed_dir = OUTPUT_DIR / "agreed-samples"
disagreed_dir = OUTPUT_DIR / "disagreed-samples"
agreed_dir.mkdir(parents=True, exist_ok=True)
disagreed_dir.mkdir(parents=True, exist_ok=True)

# (set_name, paths) for each set that was used in agreement analysis
sets_with_paths = [
    ("T5", t5_paths),
    ("TB", tb_paths),
    ("TM", tm_paths),
]
if len(tc_paths) > 0:
    sets_with_paths.append(("TC", tc_paths))

n_agreed_total = 0
n_disagreed_total = 0
for set_name, paths in sets_with_paths:
    if set_name not in agreement_results or len(paths) == 0:
        continue
    result = agreement_results[set_name]
    dis_paths = {d["path"] for d in result["disagreements"]}
    agreed_paths = [p for p in paths if p not in dis_paths]
    disagreed_paths = [d["path"] for d in result["disagreements"]]
    for p in agreed_paths[:10]:
        dest = agreed_dir / f"agreed_{set_name}_{Path(p).name}"
        shutil.copy2(p, dest)
        n_agreed_total += 1
    for p in disagreed_paths[:10]:
        dest = disagreed_dir / f"disagreed_{set_name}_{Path(p).name}"
        shutil.copy2(p, dest)
        n_disagreed_total += 1

print(f"Copied {n_agreed_total} agreed samples to: {agreed_dir}")
print(f"Copied {n_disagreed_total} disagreed samples to: {disagreed_dir}")

Copied 40 agreed samples to: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/agreed-samples
Copied 35 disagreed samples to: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/disagreed-samples


## 13. Summary and Next Steps

In [19]:
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(f"\nRun: {run_name}")
print(f"Output: {OUTPUT_DIR}")

print(f"\nPyTorch models: {OUTPUT_DIR / 'models'}")
print(f"ONNX models: {OUTPUT_DIR / 'onnx'}")

print("\n" + "-" * 60)
print("To deploy to app:")
print("-" * 60)
print(f"cp {OUTPUT_DIR / 'onnx'}/ensemble_model_1.onnx <repo>/assets/binary_ensemble/")
print(f"cp {OUTPUT_DIR / 'onnx'}/ensemble_model_2.onnx <repo>/assets/binary_ensemble/")
print(f"cp {OUTPUT_DIR / 'onnx'}/ensemble_model_3.onnx <repo>/assets/binary_ensemble/")

print("\n" + "-" * 60)
print("Performance Summary:")
print("-" * 60)
if "TM" in results:
    r = results["TM"]
    print(f"TM (merged test): Acc={r['accuracy']*100:.2f}%, Sens={r['sensitivity']*100:.2f}%, "
          f"Spec={r['specificity']*100:.2f}%, AUROC={r['auroc']:.4f}")

if "TM" in agreement_results:
    ar = agreement_results["TM"]
    print(f"Agreement with 5-class: {ar['agreement_rate']:.1f}% ({ar['agreements']}/{ar['total']})")


TRAINING COMPLETE

Run: improved_run_20260205_150642
Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642

PyTorch models: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/models
ONNX models: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/onnx

------------------------------------------------------------
To deploy to app:
------------------------------------------------------------
cp /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/onnx/ensemble_model_1.onnx <repo>/assets/binary_ensemble/
cp /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ense

In [29]:
# Sanity check: binary ensemble outputs and sample predictions (2 classes)
# - Prints the output tensor shape for one binary ONNX model
# - Runs a few sample inferences (if `paths` and `labels` are already loaded in scope)
#   to show probabilities for [non-cancer, cancer]

import numpy as np
import onnxruntime as ort
from pathlib import Path

# Use the ONNX directory from this run (fallback to previous definition if present)
binary_onnx_dir = onnx_dir if 'onnx_dir' in locals() else OUTPUT_DIR / 'onnx'
model_path = binary_onnx_dir / 'ensemble_model_1.onnx'
print(f"Using binary model: {model_path}")

sess = ort.InferenceSession(str(model_path))
output = sess.get_outputs()[0]
print("Output name:", output.name)
print("Output shape:", list(output.shape))
print("Expected classes: [non-cancer, cancer]\n")

# Run a few sample predictions if paths/labels are available
BINARY_LABEL_NAMES = ["non-cancer", "cancer"]  # index 0 and 1
if 'paths' in locals() and len(paths) > 0:
    n = min(3, len(paths))
    sample_paths = paths[:n]
    has_labels = 'labels' in locals() and len(labels) >= n
    if has_labels:
        sample_labels = labels[:n]
    for i, p in enumerate(sample_paths):
        x = preprocess_for_onnx(p)
        logits = sess.run(None, {sess.get_inputs()[0].name: x})[0]
        probs = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs = probs / probs.sum(axis=1, keepdims=True)
        print(f"Sample: {Path(p).name}")
        print(f"  P(non-cancer)={probs[0,0]:.4f}, P(cancer)={probs[0,1]:.4f}")
        if has_labels:
            expected = int(sample_labels[i])
            print(f"  Expected label: {BINARY_LABEL_NAMES[expected]} ({expected})")
        print()
else:
    print("No sample paths available in scope; run the evaluation/agreement cells first (T5/TB/TC) so paths and labels are set.")

Using binary model: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/improved_binary_ensemble/improved_run_20260205_150642/onnx/ensemble_model_1.onnx
Output name: output
Output shape: ['batch_size', 2]
Expected classes: [non-cancer, cancer]

Sample: R-103-01.jpg
  P(non-cancer)=0.9605, P(cancer)=0.0395
  Expected label: non-cancer (0)

Sample: S-124-01.jpg
  P(non-cancer)=0.8664, P(cancer)=0.1336
  Expected label: non-cancer (0)

Sample: S-97-01.jpg
  P(non-cancer)=0.9272, P(cancer)=0.0728
  Expected label: non-cancer (0)

